In [34]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [45]:
!pip install pyucell -q
!pip install mygene -q
!pip install gseapy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.0/596.0 kB 10.2 MB/s eta 0:00:00


In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mygene
import gseapy as gp

import pyucell as uc
import scanpy as sc

In [37]:
gene_degree = pd.read_csv('/content/drive/MyDrive/SSN/lioness_gene_degree_matrix.csv.gz', index_col=0)
print(gene_degree.shape)
print(gene_degree.head(3))
print(gene_degree.tail(3))

(18395, 517)


,CS0001,CS0005,CS0008,CS0009,CS0010,CS0019,CS0023,CS0046,CS0049,CS0050,...,I0341,I0521,I0467,I0125,I0525,I0338,I0524,I0251,I0276,I0221
ENSG00000000419,0,47,48,1,3,1,1,0,0,0,...,0,2,0,11,0,1,3,0,0,54
ENSG00000002746,2,0,279,0,5,3,1,0,1,2,...,1,0,0,2,1,46,0,0,53,0
ENSG00000001167,6,7,1,0,23,2,6,0,0,0,...,0,2,0,0,2,0,1,0,0,0
ENSG00000001630,7,85,0,0,0,0,33,0,2,33,...,35,9,8,1,0,0,0,0,3,2
ENSG00000001626,0,2,2,1,1,1,1,0,2,0,...,2,0,2,3,3,1,4,0,0,1


# Padronização

In [39]:
ids = gene_degree.index.tolist()

mg = mygene.MyGeneInfo()

result = mg.querymany(ids,
                      scopes='ensembl.gene',
                      fields='symbol',
                      species='human',
                      as_dataframe=True)

result.head(3)

INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-8000 ...
INFO:biothings.client:querying 8001-9000 ...
INFO:biothings.client:querying 9001-10000 ...
INFO:biothings.client:querying 10001-11000 ...
INFO:biothings.client:querying 11001-12000 ...
INFO:biothings.client:querying 12001-13000 ...
INFO:biothings.client:querying 13001-14000 ...
INFO:biothings.client:querying 14001-15000 ...
INFO:biothings.client:querying 15001-16000 ...
INFO:biothings.client:querying 16001-17000 ...
INFO:biothings.client:querying 17001-18000 ...
INFO:biothings.client:querying 18001-18395 ...
INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing quer

,_id,_score,symbol,notfound
query,,,,
ENSG00000000419,8813,32.917847,DPM1,NaN
ENSG00000002746,23072,32.917847,HECW1,NaN
ENSG00000001167,4800,32.917847,NFYA,NaN


In [40]:
gene_degree['symbol'] = gene_degree.index.map(result['symbol'].to_dict())

gene_degree = gene_degree.dropna(subset='symbol')
gene_degree = gene_degree.drop_duplicates(subset='symbol')

gene_degree.set_index('symbol', inplace=True)
gene_degree.head()

,CS0001,CS0005,CS0008,CS0009,CS0010,CS0019,CS0023,CS0046,CS0049,CS0050,...,I0341,I0521,I0467,I0125,I0525,I0338,I0524,I0251,I0276,I0221
symbol,,,,,,,,,,,,,,,,,,,,,
DPM1,0,47,48,1,3,1,1,0,0,0,...,0,2,0,11,0,1,3,0,0,54
HECW1,2,0,279,0,5,3,1,0,1,2,...,1,0,0,2,1,46,0,0,53,0
NFYA,6,7,1,0,23,2,6,0,0,0,...,0,2,0,0,2,0,1,0,0,0
CYP51A1,7,85,0,0,0,0,33,0,2,33,...,35,9,8,1,0,0,0,0,3,2
CFTR,0,2,2,1,1,1,1,0,2,0,...,2,0,2,3,3,1,4,0,0,1


# Limpeza

In [41]:
print(gene_degree.shape)

# Caso exista qualquer valor diferente de zero, o gene é mantido
gene_degree = gene_degree.loc[ (gene_degree!=0).any(axis=1) ]
print(gene_degree.shape)

(18391, 517)
(18033, 517)


## Transformação para reduzir empates

In [43]:
gene_degree = np.log2(gene_degree + 1)

gene_degree.head(3)

,CS0001,CS0005,CS0008,CS0009,CS0010,CS0019,CS0023,CS0046,CS0049,CS0050,...,I0341,I0521,I0467,I0125,I0525,I0338,I0524,I0251,I0276,I0221
symbol,,,,,,,,,,,,,,,,,,,,,
DPM1,0.000000,5.584963,5.614710,1.0,2.000000,1.000000,1.000000,0.0,0.0,0.000000,...,0.0,1.584963,0.0,3.584963,0.000000,1.000000,2.0,0.0,0.000000,5.78136
HECW1,1.584963,0.000000,8.129283,0.0,2.584963,2.000000,1.000000,0.0,1.0,1.584963,...,1.0,0.000000,0.0,1.584963,1.000000,5.554589,0.0,0.0,5.754888,0.00000
NFYA,2.807355,3.000000,1.000000,0.0,4.584963,1.584963,2.807355,0.0,0.0,0.000000,...,0.0,1.584963,0.0,0.000000,1.584963,0.000000,1.0,0.0,0.000000,0.00000


Quanto maior o valor, maior o grau do gene indicado

# Procurando Gene Sets

In [50]:
gene_sets = gp.get_library(name='KEGG_2021_Human', organism='Human')
print(list(gene_sets.keys())[:3])

['ABC transporters', 'AGE-RAGE signaling pathway in diabetic complications', 'AMPK signaling pathway']
